In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [4]:
data = pd.read_csv("Titanic.csv")
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# preprocessing and feature engineering
X = data.drop("Survived",axis =1)
y = data["Survived"]
X.drop("Name",axis=1,inplace = True)
# similarly the ticket infor seems to difficult to provide useful information out of so dropping 
X.drop("Ticket",axis=1,inplace = True)
X.drop("PassengerId",axis=1,inplace = True)
mean_age = X["Age"].mean()
X["Age"] = X["Age"].fillna(mean_age)
print(X)
X["Has_Cabin"]=  X["Cabin"].notna().astype(int)
X.drop("Cabin",axis=1,inplace=True)

embarked_map={
    "S":1,
    "C":2,
    "Q":3
}
X["Embarked"] = (
    X["Embarked"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map(embarked_map)
    .fillna(0)
)

sex_map ={
    "male":1,
    "female":2
}
X["Sex"]= (
    X["Sex"].astype(str)
    .str.strip()
    .str.lower()
    .map(sex_map)
    .fillna(0)
)

X.head()

     Pclass     Sex        Age  SibSp  Parch     Fare Cabin Embarked
0         3    male  22.000000      1      0   7.2500   NaN        S
1         1  female  38.000000      1      0  71.2833   C85        C
2         3  female  26.000000      0      0   7.9250   NaN        S
3         1  female  35.000000      1      0  53.1000  C123        S
4         3    male  35.000000      0      0   8.0500   NaN        S
..      ...     ...        ...    ...    ...      ...   ...      ...
886       2    male  27.000000      0      0  13.0000   NaN        S
887       1  female  19.000000      0      0  30.0000   B42        S
888       3  female  29.699118      1      2  23.4500   NaN        S
889       1    male  26.000000      0      0  30.0000  C148        C
890       3    male  32.000000      0      0   7.7500   NaN        Q

[891 rows x 8 columns]


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Has_Cabin
0,3,1,22.0,1,0,7.2500,1.0,0
1,1,2,38.0,1,0,71.2833,2.0,1
2,3,2,26.0,0,0,7.9250,1.0,0
3,1,2,35.0,1,0,53.1000,1.0,1
4,3,1,35.0,0,0,8.0500,1.0,0


In [ ]:
X = X.to_numpy()
y= y.to_numpy()


In [7]:
class Boosting:
    def __init__(self,iterations=16):
        self.iterations=iterations
        self.alphas=[]
        self.trees=[]

    def fit(self,X,y):
        y=np.where(y<=0,-1,1)
        m,n=X.shape
        w=np.full(m,(1/m))
        for _ in range(self.iterations):
            best_err=float('inf')
            best_feat,best_thresh,best_p=None,None,1
            for i in range(n):
                thresholds=np.unique(X[:,i])
                for thresh in thresholds:
                    for p in [1,-1]:
                        preds=np.ones(m)
                        if p==1:preds[X[:,i]<=thresh]=-1
                        else:preds[X[:,i]<=thresh]=1
                        err=np.sum(w[y!=preds])
                        if err<best_err:
                            best_err=err
                            best_feat=i
                            best_thresh=thresh
                            best_p=p
            eps=1e-15
            imp=0.5*np.log((1-best_err+eps)/(best_err+eps))
            self.trees.append((best_feat,best_thresh,best_p))
            self.alphas.append(imp)
            curr_preds=np.ones(m)
            if best_p==1:curr_preds[X[:,best_feat]<=best_thresh]=-1
            else:curr_preds[X[:,best_feat]<=best_thresh]=1
            w*=np.exp(-imp*y*curr_preds)
            w/=np.sum(w)

    def predict(self,X):
        m=X.shape[0]
        f_preds=np.zeros(m)
        for i in range(len(self.trees)):
            feat,thresh,p=self.trees[i]
            alpha=self.alphas[i]
            s_preds=np.ones(m)
            if p==1:s_preds[X[:,feat]<=thresh]=-1
            else:s_preds[X[:,feat]<=thresh]=1
            f_preds+=alpha*s_preds
        return np.where(f_preds>=0,1,0)

In [28]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2
)
base = Boosting()
base.fit(X_train,y_train)
y_pred = base.predict(X_test)
print(f"the boosted acc is {np.mean(y_test== y_pred)*100:.2f}%")

the boosted acc is 82.68%
